In [ ]:
'''
Frozen-ML PAI ablations on LLM evaluation datasets.
'''
import torch
from torch.nn.functional import logsigmoid
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
from IPython.display import clear_output
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 100, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# which dataset are we running?
dataset = "mmlu-pro" # or "bbh+gpqa+ifeval+math+musr"

# filename for this simulation
fname = f"{dataset}_pai_ablation.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "model_idx", 
           "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")
        
# dummy tensors for brevity
ZERO = torch.tensor(0.0, device=device)

In [ ]:
#### LOAD IN OUR DATA

# load in initial factor model fits for LLM evaluations
U = torch.load(f"../datasets/llm_evals/{dataset}/U.pt", map_location=device).to(device) # (N_OLD, D)
V = torch.load(f"../datasets/llm_evals/{dataset}/V.pt", map_location=device).to(device) # (N_Q, D)
ys = torch.load(f"../datasets/llm_evals/{dataset}/Y_test.pt", map_location=device).to(device) # (N_NEW, N_Q)

# get our finite-population size + no. of new models
N_NEW, N_Q = ys.shape; D = U.shape[1]

In [ ]:
#### PRO-ACTIVE INFERENCE SAMPLING

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # how many labels can we collect?
    n = int(budget * N_Q)
    
    # for LLM evaluations, we are parallelizing across models. So need for-loop over seeds
    for seed in range(N_SEEDS):
        
        # set a seed for reproducibility
        torch.random.manual_seed(858 + seed)
        
        #### initialize data structures
        
        # data structure of which samples have already been observed?
        already_sampled = torch.zeros(size=(N_NEW, N_Q), dtype=bool, device=device) # (N_NEW, N_Q)

        # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by n) and sigmahat^2 (need div by nN^2)
        thetahats_unnormalized = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        sigmahats_sq_unnormalized = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        
        # initialize new model factors
        Uhats = torch.tile(U.mean(axis=0, keepdim=True), (N_NEW, 1)) # (N_NEW, D)
        diag_precisions = torch.tile(1.0 / (U.std(axis=0, keepdim=True) ** 2), (N_NEW, 1)) # (N_NEW, D)

        #### adaptive sampling rounds

        # loop thru our budget
        for t in tqdm(range(n), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}, Seed {seed} of {N_SEEDS}"):

            #### PART 1. computing q_t(j) probabilities + sampling probabilities

            # a. compute the uncertainty scores based on the current ML prediction
            logits = Uhats @ V.T # (N_NEW, N_Q)
            fxs = torch.sigmoid(logits) # (N_NEW, N_Q)

            # b. more stable: min(\sig(z), 1 - \sig(z)) = min(\sig(z), \sig(-z)) = \sig(min(z, -z)) = \sig(-|z|)
            log_u_ts = logsigmoid(-logits.abs()) # (N_NEW, N_Q)

            # c. zero out previously observed entries
            log_u_ts = log_u_ts.masked_fill(already_sampled, -torch.inf) # (N_NEW, N_Q)

            # d. safe normalization from log-scale to original scale + mix with tau-governed uniform
            h_js = torch.softmax(log_u_ts, dim=1) # (N_NEW, N_Q)
            q_js = (TAU * ~already_sampled / (N_Q - t)) + ((1.0 - TAU) * h_js) # (N_NEW, N_Q)

            # e. which datapoint are we sampling?
            I_t = torch.multinomial(input=q_js, num_samples=1) # (N_NEW, 1)

            #### PART 2: Updating Our PAI Estimators

            # a. get the queried values + build AIPW correction
            y_Is = torch.gather(ys, dim=1, index=I_t) # (N_SEEDS, 1)
            f_Is = torch.gather(fxs, dim=1, index=I_t) # (N_NEW, 1)
            q_Is = torch.gather(q_js, dim=1, index=I_t) # (N_NEW, 1)
            aipw_correction_Is = (y_Is - f_Is) / q_Is

            # b. building the Ahat_n (without the / nN^2)
            sigmahats_sq_unnormalized += (aipw_correction_Is ** 2) # first term

            # c. building the Bhat_n (also without the / nN^2)
            if t > 0:
                t2 = (N_Q * thetahats_unnormalized / t)\
                - (ys * already_sampled).sum(dim=1, keepdim=True)\
                - (fxs * ~already_sampled).sum(dim=1, keepdim=True)
                sigmahats_sq_unnormalized -= (t2 ** 2)

            # d. add in the bias-correction term Chat_n for all but the last term (also without the / nN^2)
            if t < n - 1:
                t3 = (aipw_correction_Is ** 2)
                t3 *= ((1.0 / torch.arange(t+1, n, device=device, dtype=torch.float32)) ** 2).sum()
                sigmahats_sq_unnormalized += t3

            # e. compute our phi_t and increment our estimator (UNNORMALIZED)
            phi_ts = (
                (fxs * ~already_sampled).sum(dim=1, keepdim=True) +\
                (ys * already_sampled).sum(dim=1, keepdim=True) +\
                aipw_correction_Is
            ) / N_Q
            thetahats_unnormalized += phi_ts

            # f. update our observed indices
            already_sampled.scatter_(dim=1, index=I_t, src=torch.full_like(I_t, fill_value=True, dtype=bool))
            
            '''
            FOR THIS ABLATION, WE ARE TURNING OFF ONLINE MODEL UPDATES!
            
            #### PART 3: updating our online logistic regression

            # fhat * (1 - fhat); note that diag(xx^T) = (x_1^2, \dots, x_D^2)
            w_Is = f_Is * (1.0 - f_Is) # (N_NEW, 1)
            X_Is = V[I_t.squeeze()] # (N_NEW, D)
            diag_precisions += w_Is * (X_Is ** 2) # (N_SEEDS, D)
            Uhats += (y_Is - f_Is) * X_Is / diag_precisions
            '''

        #### PART 4: combining + normalizing to make PAI estimates

        # get our mean estimate + variance estimate
        thetahats = thetahats_unnormalized / n
        sigmahats_sq = torch.maximum(sigmahats_sq_unnormalized, ZERO) / (n * (N_Q ** 2))

        # get our z-score + the lower/upper limits of our interval
        z_score = norm.ppf(1 - (ALPHA / 2))
        ubs = thetahats + z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        lbs = thetahats - z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        widths = (ubs - lbs).cpu().numpy()
        coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()

        # for file writing purposes, convert thetahats
        thetahats = thetahats.cpu().numpy()

        # format results + directly write to file
        results = [
            [dataset, budget, TAU, "pai", model_idx, seed, 
             thetahats[model_idx][0], coverages[model_idx][0], widths[model_idx][0], n
            ] for model_idx in range(N_NEW)]
        with open(f"../logs/{fname}", "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerows(results)
            
        # clean house
        clear_output(wait=True)